# Day3_02D_Context_Aware_Multi_Agent

## OpenAI Agents SDK - Advanced Agent Pattern

**Duration:**20 minutes

### Learning Objectives
- Understand what context means in Agentic AI.
- Pass user context safely between agents.
- Personalize responses based on role.
- Build context-aware multi-agent workflows.
- Apply the concept to universities and enterprises.

## Why Context Matters

Two users ask the same question:

> Explain Agentic AI.

A first-year student needs a simple explanation.
A Professor needs a deeper explanation.
A Principal needs implementation strategy.

The question is identical.
The context is different.

Context allows the same Agent to adapt intelligently.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from dataclasses import dataclass
from typing import List
from agents import Agent, Runner, RunContextWrapper

current=Path.cwd().resolve()
for folder in [current]+list(current.parents):
    env=folder/".env"
    if env.exists():
        load_dotenv(env)
        break

print("API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)


## Create Context Model

In [ ]:
@dataclass
class UserContext:
    name:str
    role:str
    department:str
    interests:List[str]


## Create Context-Aware Agent

In [ ]:
faculty_agent = Agent(
    name="DSU Faculty Assistant",
    instructions="""
    You are a teaching assistant.
    Use the supplied user context to personalise your response.
    Adjust technical depth according to the user's role.
    """,
)

## Student Context

In [ ]:
student = UserContext(
    name="Rahul",
    role="Student",
    department="AI & ML",
    interests=["Python","Agentic AI"]
)

result = await Runner.run(
    faculty_agent,
    "Explain Agentic AI.",
    context=student
)

print(result.final_output)


## Faculty Context

In [ ]:
faculty = UserContext(
    name="Dr. Meena",
    role="Professor",
    department="CSE",
    interests=["Research","LLMs"]
)

result = await Runner.run(
    faculty_agent,
    "Explain Agentic AI.",
    context=faculty
)

print(result.final_output)


### What did we just learn?
The question remained the same, but the response should adapt to the supplied context.

## Multi-Agent with Shared Context

In [ ]:
research_agent = Agent(
    name="Research Advisor",
    instructions="Provide research-oriented guidance using the supplied context."
)

lab_agent = Agent(
    name="Lab Instructor",
    instructions="Suggest practical lab activities using the supplied context."
)

In [ ]:
async def prepare_personalized_session(topic:str, ctx:UserContext):

    research = await Runner.run(
        research_agent,
        f"Provide research guidance on {topic}",
        context=ctx
    )

    lab = await Runner.run(
        lab_agent,
        f"Suggest one practical lab on {topic}",
        context=ctx
    )

    return f'''
## Research Guidance

{research.final_output}

---

## Lab Activity

{lab.final_output}
'''

In [ ]:
output = await prepare_personalized_session(
    "Agentic AI",
    faculty
)

print(output)


## Enterprise Example

Customer asks:
> Explain my loan eligibility.

The response depends on context:

- Customer type
- Credit score
- Existing loans
- Country
- Regulatory policy

Without context, personalization is impossible.


## Context vs Memory

| Context | Memory |
|---|---|
| Supplied for the current run | Built over multiple interactions |
| Explicit | Learned over time |
| Temporary | Persistent |

Context answers **Who is the user right now?**

Memory answers **What have I learned about the user over time?**


## Hands-on Exercise

Create three contexts:

- Student
- Faculty
- HOD

Ask the same question:

> Explain RAG.

Observe how the response changes.

Then create a Placement Officer context and ask:

> Prepare placement training on Agentic AI.


## Key Takeaways

- Context makes Agent responses personalized.
- The same Agent can behave differently for different users.
- Context can include role, department, permissions, preferences, or business data.
- Multi-Agent systems can share the same context across specialist Agents.
- Context is different from long-term memory.

### Final Mental Model

User + Context → Agent(s) → Personalized Response
